In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator

In [ ]:
# Helper Functions
def pca_func(data: pd.DataFrame, title_suffix: str = "") -> tuple[PCA, pd.DataFrame]:
    """
    Perform PCA on a dataset and plot explained variance.
    Returns the fitted PCA object and PCA score DataFrame.
    """
    pca = PCA(n_components=data.shape[1])
    scores = pca.fit_transform(data)

    # Explained variance plot
    plt.figure(figsize=(6, 4))
    plt.plot(pca.explained_variance_, marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()


    # Print explained variance table
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)

    # Return PCA model and scores DataFrame
    df_scores = pd.DataFrame(
        scores,
        columns=[f'PC{i}' for i in range(1, scores.shape[1] + 1)],
        index=data.index
    )
    return pca, df_scores


In [ ]:
def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,
           pcx: int = 1, pcy: int = 2, new_axes: bool = False,
           title: str | None = None) -> None:
    """
    Create a PCA biplot for given principal components.
    """
    x_label = f"PC{pcx}"
    y_label = f"PC{pcy}"

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(df_scores[x_label], df_scores[y_label], color="b", alpha=0.7)

    # Axis labels
    if new_axes:
        explvar = 100 * pca.explained_variance_ratio_
        ax.set_xlabel(f"{x_label} ({explvar[pcx-1]:.1f}% explained var.)", fontsize=10)
        ax.set_ylabel(f"{y_label} ({explvar[pcy-1]:.1f}% explained var.)", fontsize=10)
    else:
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)

    # Equal aspect for scores
    ax.set_aspect('equal', adjustable='datalim')

    # Create twin axes for loadings
    ax2 = ax.twinx().twiny()
    font = {'color': 'g', 'weight': 'bold', 'size': 10}

    for col in df_loadings.columns:
        tipx = df_loadings.loc[x_label, col]
        tipy = df_loadings.loc[y_label, col]
        ax2.arrow(0, 0, tipx, tipy, color="r", alpha=0.5, length_includes_head=True)
        ax2.text(tipx * 1.05, tipy * 1.05, col, fontdict=font, ha="center", va="center")

    # Align axes centers
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)

    # Keep loading axes square
    ax2.set_aspect('equal', adjustable='datalim')

    if title:
        plt.title(title)
    plt.tight_layout()

In [ ]:
# -----------------------------
# Main Analysis
# -----------------------------

# Load and prepare data
dfr = pd.read_csv("data/humor_data_clean.csv", index_col=0)
df = dfr

# Standardize
dfs = pd.DataFrame(
    StandardScaler().fit_transform(df),
    columns=df.columns,
    index=df.index
)


In [ ]:

# PCA
pca, df_scores = pca_func(dfs, " (Humor Data)")
df_loadings = pd.DataFrame(
    pca.components_,
    columns=dfs.columns,
    index=df_scores.columns
)

# -----------------------------
# Biplots for selected component pairs
# -----------------------------
biplot(df_scores, df_loadings, pca, pcx=1, pcy=2, new_axes=True, title="Humor PCA Biplot (PC1 vs PC2)")
biplot(df_scores, df_loadings, pca, pcx=1, pcy=3, new_axes=True, title="Humor PCA Biplot (PC1 vs PC3)")
biplot(df_scores, df_loadings, pca, pcx=1, pcy=4, new_axes=True, title="Humor PCA Biplot (PC1 vs PC4)")
plt.show()

In [ ]:
# PCA with Varimax Rotation
# -----------------------------
k=4
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(dfs)

loadings_df = pd.DataFrame(pca.components_.T, index=df.columns, columns=["PC1", "PC2", "PC3", "PC4"])
print(loadings_df)

rotator = Rotator(method="varimax")
loadings_rot = rotator.fit_transform(pca.components_.T)

R = rotator.rotation_

load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["RPC1", "RPC2", "RPC3", "PC4"])
print(load_rot_df)

In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than this value (cut),  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)

print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))
